# Synthea CSV integration: complete walkthrough

This notebook exercises the complete CSV-only integration:

- configuration discovery before construction;
- every `SyntheaGenerator` constructor argument;
- every public generator method;
- representative `synthea.properties` overrides;
- generation and loading through `SyntheaCSVDataset`; and
- an equivalence check against a direct Synthea Java invocation using the same version, parameters, configuration, and seeds.

The equivalence check asserts emitted CSV file names and ordered column names. Row counts, patient IDs, and byte hashes are reported as stronger diagnostics without being required to match across a published release binary and a fresh local source build.

## Prerequisites

Java 17 or newer, Git, and internet access must be available. PyHealth downloads and checksum-verifies its pinned Synthea JAR on first use unless `jar_path` is supplied. The reference path separately clones the official Synthea repository and builds/runs its release tag through Gradle, so the live cells can take several minutes.

In [2]:
import csv
import hashlib
import inspect
import shlex
import shutil
import subprocess
import tempfile
from pathlib import Path

from pyhealth.datasets import SyntheaCSVDataset, SyntheaGenerator

In [3]:
artifact_root = Path(".artifacts/synthea_csv").resolve()
artifact_root.mkdir(parents=True, exist_ok=True)
artifact_root

PosixPath('/home/soham/PyHealth/examples/.artifacts/synthea_csv')

## Public generator API

| Method | Purpose |
|---|---|
| `get_available_config()` | Return properties supported by the selected Synthea JAR |
| `output_path()` | Return the configured CSV directory |
| `resolved_output_path()` | Resolve a nested folder-per-run CSV directory when present |
| `with_config()` | Return a new generator with merged property overrides |
| `build_argv()` | Build the Java/Synthea argument vector without executing it |
| `run()` | Execute Synthea and validate that CSV output was produced |
| `ensure_generated()` | Reuse existing output or generate it when absent |

In [ ]:
expected_public_methods = {
    "get_available_config",
    "output_path",
    "resolved_output_path",
    "with_config",
    "build_argv",
    "run",
    "ensure_generated",
}
assert all(hasattr(SyntheaGenerator, name) for name in expected_public_methods)
{
    name: str(inspect.signature(getattr(SyntheaGenerator, name)))
    for name in sorted(expected_public_methods)
}

## 1. Discover supported Synthea configuration

These class methods work before a generator exists. `pattern` uses shell-style glob matching. Discovery reads `synthea.properties` from the actual pinned or explicitly supplied JAR, so it reflects the Synthea version that will run.

In [5]:
generation_properties = SyntheaGenerator.get_available_config(
    pattern="generate.*"
)
list(sorted(generation_properties.items()))[:10]

[('generate.append_numbers_to_person_names', 'true'),
 ('generate.birthweights.default_file', 'birthweights.csv'),
 ('generate.birthweights.logging', 'false'),
 ('generate.costs.default_device_cost', '0.00'),
 ('generate.costs.default_encounter_cost', '125.00'),
 ('generate.costs.default_immunization_cost', '136.00'),
 ('generate.costs.default_lab_cost', '100.00'),
 ('generate.costs.default_medication_cost', '255.00'),
 ('generate.costs.default_procedure_cost', '500.00'),
 ('generate.costs.default_supply_cost', '0.00')]

The examples below use several ordinary Synthea properties:

- `generate.thread_pool_size=1` makes the equivalence run deterministic and serial;
- `generate.only_alive_patients=true` keeps only patients alive at the simulation end;
- `exporter.years_of_history=5` limits exported history;
- `exporter.csv.folder_per_run=false` keeps CSV files directly under the CSV directory; and
- `exporter.csv.append_mode=false` replaces rather than appends to prior CSV files.

CSV exporter selection itself is managed by the wrapper and should not be placed in `synthea_config`.

## 2. Construct a normal runnable generator

In [7]:
common_config = {
    "generate.thread_pool_size": 1,
    "generate.only_alive_patients": True,
    "exporter.years_of_history": 5,
    "exporter.csv.folder_per_run": False,
    "exporter.csv.append_mode": False,
}

generator = SyntheaGenerator(
    output_dir=artifact_root / "generated",
    population=10,
    seed=42,
    clinician_seed=43,
    reference_date="20240101",
    gender="F",
    age_range="30-40",
    overflow_population=False,
    state="Massachusetts",
    city="Boston",
    synthea_config=common_config,
    timeout=900,
)
generator

`output_path()` is the expected CSV location. `resolved_output_path()` initially returns the same path; if folder-per-run output is enabled, it locates the actual nested directory after generation. `build_argv()` is useful for inspection and testing and does not execute Java.

In [8]:
expected_csv_path = generator.output_path()
resolved_before_generation = generator.resolved_output_path()
preview_argv = generator.build_argv(Path("java"), Path("synthea.jar"))

print("Expected CSV path:", expected_csv_path)
print("Resolved path now:", resolved_before_generation)
print("Command preview:")
print(shlex.join(str(item) for item in preview_argv))

Expected CSV path: /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv
Resolved path now: /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv
Command preview:
java -jar synthea.jar --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_histor

## 3. Update configuration without mutating the original

`with_config()` returns another generator, re-runs constructor validation, and recomputes derived state such as the fingerprinted generation directory.

In [15]:
updated_generator = generator.with_config({
    "generate.only_alive_patients": False,
    "exporter.years_of_history": 10,
})

assert generator.synthea_config["generate.only_alive_patients"] == "true"
assert updated_generator.synthea_config["generate.only_alive_patients"] == "false"
assert updated_generator.generation_dir != generator.generation_dir

{
    "original": generator.generation_dir,
    "updated": updated_generator.generation_dir,
}

{'original': PosixPath('/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4'),
 'updated': PosixPath('/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/69d4854b026bb54c817298b85df3947987f2d1c3921a4ea6c3e04631e2b863e2')}

## 4. Every constructor argument

The following generator is intentionally a **dry-run API example**. Snapshot, fixed-record, keep-matching, and custom-module inputs require domain-specific files, so this object is used only to prove argument coverage and inspect its command. Do not call `run()` on it until those placeholder files are replaced with valid Synthea inputs.

Passing `None` means that option is omitted and Synthea uses its own default. When both `local_config_path` and `synthea_config` define a property, the mapping is emitted later as a command-line override and therefore wins.

In [16]:
constructor_demo_root = artifact_root / "constructor_demo"
constructor_demo_root.mkdir(parents=True, exist_ok=True)
local_config_path = constructor_demo_root / "local.properties"
local_config_path.write_text(
    "generate.only_alive_patients = true\n"
    "exporter.years_of_history = 5\n",
    encoding="utf-8",
)

all_constructor_args = {
    "output_dir": constructor_demo_root / "output",
    "population": 25,
    "seed": 1001,
    "state": "Massachusetts",
    "city": "Boston",
    "clinician_seed": 1002,
    "single_person_seed": 1003,
    "reference_date": "20240101",
    "end_date": "20241231",
    "gender": "F",
    "age_range": "18-65",
    "overflow_population": False,
    "local_config_path": local_config_path,
    "local_modules_dir": constructor_demo_root / "modules",
    "initial_population_snapshot_path": constructor_demo_root / "initial.snapshot",
    "updated_population_snapshot_path": constructor_demo_root / "updated.snapshot",
    "update_time_period": 30,
    "fixed_record_path": constructor_demo_root / "fixed_records.json",
    "keep_matching_patients_path": constructor_demo_root / "keep_module.json",
    "java_path": None,
    "jar_path": None,
    "auto_download": True,
    "synthea_config": {
        "generate.thread_pool_size": 1,
        "generate.only_alive_patients": False,
        "exporter.csv.folder_per_run": False,
        "exporter.csv.append_mode": False,
    },
    "timeout": 900.0,
    "regenerate": False,
}

declared_args = set(inspect.signature(SyntheaGenerator).parameters)
assert set(all_constructor_args) == declared_args
all_args_generator = SyntheaGenerator(**all_constructor_args)
all_args_argv = all_args_generator.build_argv(Path("java"), Path("synthea.jar"))
assert all_args_argv.index("-c") < all_args_argv.index("--generate.only_alive_patients=false")
print(shlex.join(all_args_argv))

java -jar synthea.jar -c /home/soham/PyHealth/examples/.artifacts/synthea_csv/constructor_demo/local.properties --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/constructor_demo/output/5edeeb5fa66bd2d663d3a052a9566c4eccb860b6ec5831c3ed16f6345c6f020a --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --generate.only_alive_patients=false --generate.thread_pool_size=1 -s 1001 -cs 1002 -ps 1003 -p 25 -r 20240101 -e 20241231 -g F -a 18-65 -o false -d /home/soham/PyHealth/examples/.artifacts/synthea_csv/constructor_demo/modules -i /home/soham

Constructor-to-CLI mapping:

| Python argument | Synthea CLI | Meaning |
|---|---|---|
| `seed` | `-s` | Population random seed |
| `clinician_seed` | `-cs` | Clinician generation seed |
| `single_person_seed` | `-ps` | Seed used to reproduce a particular person |
| `population` | `-p` | Requested population size |
| `reference_date` | `-r` | Reference date in `YYYYMMDD` form |
| `end_date` | `-e` | Simulation end date in `YYYYMMDD` form |
| `gender` | `-g` | Restrict generation to `M` or `F` |
| `age_range` | `-a` | Restrict age using `min-max` |
| `overflow_population` | `-o` | Permit extra generated people to satisfy filters |
| `local_config_path` | `-c` | Additional `.properties` configuration file |
| `local_modules_dir` | `-d` | Directory containing custom Synthea modules |
| `initial_population_snapshot_path` | `-i` | Load a previously saved population snapshot |
| `updated_population_snapshot_path` | `-u` | Write the population after an update run |
| `update_time_period` | `-t` | Number of days advanced during a snapshot update |
| `fixed_record_path` | `-f` | Fixed-demographics input used to generate specified people |
| `keep_matching_patients_path` | `-k` | Module used to retain only matching patients |
| `state`, `city` | trailing positional arguments | Geographic generation location |
| `synthea_config` | `--property=value` overrides | Arbitrary supported `synthea.properties` values |
| `output_dir` | wrapper-managed | Parent of the fingerprinted generation directory |
| `java_path` | wrapper-managed | Explicit Java executable, otherwise discovered |
| `jar_path` | wrapper-managed | Explicit Synthea JAR, otherwise the pinned release is used |
| `auto_download` | wrapper-managed | Allow downloading the pinned JAR when absent |
| `timeout` | wrapper-managed | Subprocess timeout in seconds |
| `regenerate` | wrapper-managed | Replace existing output on the first lazy access |

## 5. Generate, validate, and reuse CSV output

`run()` always executes Synthea. `ensure_generated()` is the normal lazy path: it generates only when `patients.csv` is absent, unless `regenerate=True`.

In [17]:
csv_path = generator.run()
reused_path = generator.ensure_generated()
resolved_after_generation = generator.resolved_output_path()

assert csv_path == reused_path == resolved_after_generation
assert (csv_path / "patients.csv").is_file()
sorted(path.name for path in csv_path.glob("*.csv"))

Running Synthea: /home/soham/.local/jvm/jdk-17.0.20.1+1/bin/java -jar /home/soham/.cache/pyhealth/synthea/synthea-with-dependencies-v4.0.0.jar --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_history=5 --generate.only_alive_patients=true --generate.thread_pool_size=1 -s 42 -cs 43 -p 10 -r 20240101 -g F -a 30-40 -o false Massachusetts Boston
org.mitre.synthea.X12.ExporterAdaptor
org.mitre.synth

SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#noProviders for further details.


Scanned 92 modules and 449 submodules.
Loading submodule modules/allergies/allergy_panel.json
Loading submodule modules/allergies/drug_allergy_incidence.json
Loading submodule modules/allergies/environmental_allergy_incidence.json
Loading submodule modules/allergies/food_allergy_incidence.json
Loading submodule modules/allergies/immunotherapy.json
Loading submodule modules/allergies/outgrow_env_allergies.json
Loading submodule modules/allergies/outgrow_food_allergies.json
Loading submodule modules/allergies/severe_allergic_reaction.json
Loading submodule modules/anemia/anemia_sub.json
Loading submodule modules/breast_cancer/chemotherapy_breast.json
Loading submodule modules/breast_cancer/hormone_diagnosis.json
Loading submodule modules/breast_cancer/hormonetherapy_breast.json
Loading submodule modules/breast_cancer/surgery_therapy_breast.json
Loading submodule modules/breast_cancer/tnm_diagnosis.json
Loading submodule modules/contraceptives/clear_contraceptive.json
Loading submodule mo

['allergies.csv',
 'careplans.csv',
 'claims.csv',
 'claims_transactions.csv',
 'conditions.csv',
 'devices.csv',
 'encounters.csv',
 'imaging_studies.csv',
 'immunizations.csv',
 'medications.csv',
 'observations.csv',
 'organizations.csv',
 'patients.csv',
 'payer_transitions.csv',
 'payers.csv',
 'procedures.csv',
 'providers.csv',
 'supplies.csv']

## 6. Load generated CSV through PyHealth

`SyntheaCSVDataset` accepts the generator plus an optional table selection. Omitting `tables` uses all supported defaults. `dataset_name`, `config_path`, and normal `BaseDataset` keyword arguments such as `cache_dir` and `dev` can also be supplied.

In [18]:
dataset = SyntheaCSVDataset(
    generator=generator,
    tables=["patients", "encounters", "conditions", "observations"],
    dataset_name="synthea_csv_demo",
    config_path=None,
    cache_dir=artifact_root / "cache",
    dev=False,
)

events = dataset.load_data()
dataset.stats()
# events.head(5)
events

Initializing synthea_csv_demo dataset from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv (dev mode: False)
Using provided cache_dir: /home/soham/PyHealth/examples/.artifacts/synthea_csv/cache/52ec6deb-b3eb-515b-a2e3-c799d1d15b9a
Scanning table: patients from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv/patients.csv
Scanning table: encounters from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv/encounters.csv
Scanning table: conditions from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv/conditions.csv
Scanning table: observations from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd

,patient_id,event_type,timestamp,patients/birthdate,patients/deathdate,patients/gender,patients/race,patients/ethnicity,patients/marital,patients/birthplace,patients/city,patients/state,patients/county,patients/zip,encounters/id,encounters/start,encounters/stop,encounters/encounterclass,encounters/code,encounters/description,encounters/base_encounter_cost,encounters/total_claim_cost,encounters/payer_coverage,encounters/reasoncode,encounters/reasondescription,conditions/start,conditions/stop,conditions/encounter,conditions/code,conditions/description,observations/date,observations/encounter,observations/code,observations/description,observations/value,observations/units,observations/type
npartitions=4,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,string,object,datetime64[ms],string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [19]:
patients_table = dataset.load_table("patients")
patient_ids = dataset.unique_patient_ids
patient = dataset.get_patient(patient_ids[0])
patient_events = patient.get_events(return_df=True)

print("Patients:", len(patient_ids))
print("Selected patient:", patient.patient_id)
print("Configured tables:", dataset.tables)
patient_events.head(10)

Scanning table: patients from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/999ea78e6ea632557cee855051132f901a3b9eca55bc4c349bbaffd06ba355d4/csv/patients.csv
Found 10 unique patient IDs
Patients: 10
Selected patient: e74367bd-7385-97dc-2c7f-251b4cd9fe29
Configured tables: ['patients', 'encounters', 'conditions', 'observations']


patient_id,event_type,timestamp,patients/birthdate,patients/deathdate,patients/gender,patients/race,patients/ethnicity,patients/marital,patients/birthplace,patients/city,patients/state,patients/county,patients/zip,encounters/id,encounters/start,encounters/stop,encounters/encounterclass,encounters/code,encounters/description,encounters/base_encounter_cost,encounters/total_claim_cost,encounters/payer_coverage,encounters/reasoncode,encounters/reasondescription,conditions/start,conditions/stop,conditions/encounter,conditions/code,conditions/description,observations/date,observations/encounter,observations/code,observations/description,observations/value,observations/units,observations/type
str,str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""e74367bd-7385-97dc-2c7f-251b4c…","""patients""",1989-12-09 00:00:00,"""1989-12-09""",null,"""F""","""white""","""nonhispanic""","""M""","""Greenfield Massachusetts US""","""Boston""","""Massachusetts""","""Suffolk County""","""02115""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""encounters""",1999-12-18 06:32:37,null,null,null,null,null,null,null,null,null,null,null,"""e74367bd-7385-97dc-1f9b-b65e80…","""1999-12-18T06:32:37Z""","""1999-12-18T06:47:37Z""","""wellness""","""410620009""","""Well child visit (procedure)""","""136.80""","""488.50""","""0.00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""conditions""",2008-02-02 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2008-02-02""",null,"""e74367bd-7385-97dc-f8da-7aea5f…","""473461003""","""Educated to high school level …",null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""conditions""",2008-02-02 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2008-02-02""",null,"""e74367bd-7385-97dc-f8da-7aea5f…","""422650009""","""Social isolation (finding)""",null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""encounters""",2008-02-02 06:32:37,null,null,null,null,null,null,null,null,null,null,null,"""e74367bd-7385-97dc-f8da-7aea5f…","""2008-02-02T06:32:37Z""","""2008-02-02T07:15:34Z""","""wellness""","""162673000""","""General examination of patient…","""136.80""","""704.20""","""0.00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""conditions""",2011-01-22 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2011-01-22""",null,"""e74367bd-7385-97dc-28eb-970242…","""40055000""","""Chronic sinusitis (disorder)""",null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""encounters""",2011-01-22 01:32:37,null,null,null,null,null,null,null,null,null,null,null,"""e74367bd-7385-97dc-28eb-970242…","""2011-01-22T01:32:37Z""","""2011-01-22T01:47:37Z""","""ambulatory""","""185345009""","""Encounter for symptom (procedu…","""85.55""","""92.93""","""0.00""","""75498004""","""Acute bacterial sinusitis (dis…",null,null,null,null,null,null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""conditions""",2012-02-11 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2012-02-11""",null,"""e74367bd-7385-97dc-278c-6f3533…","""162864005""","""Body mass index 30+ - obesity …",null,null,null,null,null,null,null
"""e74367bd-7385-97dc-2c7f-251b4c…","""encounters""",2012-02-11 06:32:37,null,null,null,null,null,null,null,null,null,null,null,"""e74367bd-7385-97dc-278c-6f3533…","""2012-02-11T06:32:37Z""","""2012-02-11T07:19:23Z""","""wellness""","""162673000""","""General examination

## 7. Equivalence with direct Synthea execution

This is an independent source-level comparison. The reference path clones the official GitHub repository at the same `v4.0.0` release used by PyHealth and invokes the repository's `./run_synthea` CLI. That script runs the checked-out source through its Gradle wrapper; it does not use PyHealth's downloaded JAR.

The wrapper and cloned-repository runs use:

- two independently obtained Synthea artifacts at release `v4.0.0`;
- the same population, seeds, demographic filters, state, and city;
- the same complete effective Synthea property set; and
- different base directories so their outputs can be compared independently.

The reference command is assembled as a real `./run_synthea ...` command using the [official launcher](https://github.com/synthetichealth/synthea/blob/v4.0.0/run_synthea) and [documented CLI flags](https://github.com/synthetichealth/synthea/blob/v4.0.0/README.md#generate-synthetic-patients). The asserted contract is the set of CSV files and each file's ordered output fields. Row counts, patient IDs, and byte hashes are reported separately because a freshly compiled source checkout can generate different clinical histories from the published binary even with the same seeds.

In [20]:
synthea_repository = "https://github.com/synthetichealth/synthea.git"
synthea_tag = "v4.0.0"
synthea_tag_commit = "0185c09ea9d10a822c6f5f3ef9bdcbcbe960c813"
synthea_checkout = artifact_root / "upstream" / f"synthea-{synthea_tag}"

assert shutil.which("git"), "Git is required for the reference checkout"
assert shutil.which("java"), "Java 17+ is required by Synthea"

if not (synthea_checkout / ".git").is_dir():
    synthea_checkout.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            synthea_tag,
            "--depth",
            "1",
            synthea_repository,
            str(synthea_checkout),
        ],
        check=True,
    )

checkout_commit = subprocess.run(
    ["git", "-C", str(synthea_checkout), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert checkout_commit == synthea_tag_commit
print(f"Reference checkout: {synthea_checkout}")
print(f"Tag/commit: {synthea_tag} / {checkout_commit}")

Cloning into '/home/soham/PyHealth/examples/.artifacts/synthea_csv/upstream/synthea-v4.0.0'...
Note: switching to '0185c09ea9d10a822c6f5f3ef9bdcbcbe960c813'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



Reference checkout: /home/soham/PyHealth/examples/.artifacts/synthea_csv/upstream/synthea-v4.0.0
Tag/commit: v4.0.0 / 0185c09ea9d10a822c6f5f3ef9bdcbcbe960c813


In [21]:
def cli_scalar(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    return str(value)


def csv_contract(root):
    contract = {}
    for path in sorted(Path(root).glob("*.csv")):
        with path.open(newline="", encoding="utf-8-sig") as stream:
            reader = csv.reader(stream)
            header = next(reader, [])
            rows = sum(1 for _ in reader)
        contract[path.name] = {
            "fields": header,
            "rows": rows,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    return contract


def read_patient_ids(root):
    with (Path(root) / "patients.csv").open(
        newline="", encoding="utf-8-sig"
    ) as stream:
        return {row["Id"] for row in csv.DictReader(stream)}

In [22]:
equivalence_args = {
    "population": 10,
    "seed": 2026,
    "clinician_seed": 2027,
    "reference_date": "20240101",
    "gender": "F",
    "age_range": "30-40",
    "overflow_population": False,
    "state": "Massachusetts",
    "city": "Boston",
}
equivalence_config = {
    "generate.thread_pool_size": 1,
    "generate.only_alive_patients": True,
    "exporter.years_of_history": 5,
    "exporter.csv.folder_per_run": False,
    "exporter.csv.append_mode": False,
}

with tempfile.TemporaryDirectory(prefix="synthea_equivalence_") as tmp:
    comparison_root = Path(tmp)
    wrapper = SyntheaGenerator(
        output_dir=comparison_root / "wrapper",
        synthea_config=equivalence_config,
        timeout=900,
        **equivalence_args,
    )

    wrapper_csv = wrapper.run()

    direct_root = comparison_root / "direct"
    wrapper_preview = wrapper.build_argv(Path("java"), Path("synthea.jar"))
    direct_config = {}
    for argument in wrapper_preview:
        if argument.startswith("--"):
            key, value = argument[2:].split("=", 1)
            direct_config[key] = value
    direct_config["exporter.baseDirectory"] = str(direct_root)

    direct_argv = ["./run_synthea"]
    direct_argv.extend(
        f"--{key}={cli_scalar(value)}"
        for key, value in sorted(direct_config.items())
    )
    direct_argv.extend(["-s", str(equivalence_args["seed"])])
    direct_argv.extend(["-cs", str(equivalence_args["clinician_seed"])])
    direct_argv.extend(["-p", str(equivalence_args["population"])])
    direct_argv.extend(["-r", equivalence_args["reference_date"]])
    direct_argv.extend(["-g", equivalence_args["gender"]])
    direct_argv.extend(["-a", equivalence_args["age_range"]])
    direct_argv.extend(
        ["-o", cli_scalar(equivalence_args["overflow_population"])]
    )
    direct_argv.extend([equivalence_args["state"], equivalence_args["city"]])

    print("Official repository CLI command:")
    print(shlex.join(direct_argv))
    reference_run = subprocess.run(
        direct_argv,
        cwd=synthea_checkout,
        capture_output=True,
        text=True,
        timeout=900,
    )
    if reference_run.returncode:
        print(reference_run.stdout)
        print(reference_run.stderr)
        raise RuntimeError(
            f"Official Synthea CLI exited with {reference_run.returncode}"
        )
    print("\n".join(reference_run.stdout.splitlines()[-12:]))
    direct_csv = direct_root / "csv"

    wrapper_contract = csv_contract(wrapper_csv)
    direct_contract = csv_contract(direct_csv)

    assert wrapper_contract.keys() == direct_contract.keys()
    for filename in wrapper_contract:
        assert wrapper_contract[filename]["fields"] == direct_contract[filename]["fields"]

    same_patient_ids = (
        read_patient_ids(wrapper_csv) == read_patient_ids(direct_csv)
    )

    equivalence_report = [
        {
            "file": filename,
            "field_count": len(wrapper_contract[filename]["fields"]),
            "fields": wrapper_contract[filename]["fields"],
            "wrapper_rows": wrapper_contract[filename]["rows"],
            "source_checkout_rows": direct_contract[filename]["rows"],
            "same_row_count": (
                wrapper_contract[filename]["rows"]
                == direct_contract[filename]["rows"]
            ),
            "same_sha256": (
                wrapper_contract[filename]["sha256"]
                == direct_contract[filename]["sha256"]
            ),
        }
        for filename in wrapper_contract
    ]

print("CSV file and output-field equivalence established.")
print("Same patient IDs:", same_patient_ids)
equivalence_report

Running Synthea: /home/soham/.local/jvm/jdk-17.0.20.1+1/bin/java -jar /home/soham/.cache/pyhealth/synthea/synthea-with-dependencies-v4.0.0.jar --exporter.baseDirectory=/tmp/synthea_equivalence_5i_fsk9v/wrapper/03dc3bdb077281e2f73b2593598cd2146457d912465b3b164fae833e83528aca --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_history=5 --generate.only_alive_patients=true --generate.thread_pool_size=1 -s 2026 -cs 2027 -p 10 -r 20240101 -g F -a 30-40 -o false Massachusetts Boston
org.mitre.synthea.X12.ExporterAdaptor
org.mitre.synthea.X12.ExporterAd

SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#noProviders for further details.


Scanned 92 modules and 449 submodules.
Loading submodule modules/allergies/allergy_panel.json
Loading submodule modules/allergies/drug_allergy_incidence.json
Loading submodule modules/allergies/environmental_allergy_incidence.json
Loading submodule modules/allergies/food_allergy_incidence.json
Loading submodule modules/allergies/immunotherapy.json
Loading submodule modules/allergies/outgrow_env_allergies.json
Loading submodule modules/allergies/outgrow_food_allergies.json
Loading submodule modules/allergies/severe_allergic_reaction.json
Loading submodule modules/anemia/anemia_sub.json
Loading submodule modules/breast_cancer/chemotherapy_breast.json
Loading submodule modules/breast_cancer/hormone_diagnosis.json
Loading submodule modules/breast_cancer/hormonetherapy_breast.json
Loading submodule modules/breast_cancer/surgery_therapy_breast.json
Loading submodule modules/breast_cancer/tnm_diagnosis.json
Loading submodule modules/contraceptives/clear_contraceptive.json
Loading submodule mo

[{'file': 'allergies.csv',
  'field_count': 15,
  'fields': ['START',
   'STOP',
   'PATIENT',
   'ENCOUNTER',
   'CODE',
   'SYSTEM',
   'DESCRIPTION',
   'TYPE',
   'CATEGORY',
   'REACTION1',
   'DESCRIPTION1',
   'SEVERITY1',
   'REACTION2',
   'DESCRIPTION2',
   'SEVERITY2'],
  'wrapper_rows': 13,
  'source_checkout_rows': 13,
  'same_row_count': True,
  'same_sha256': True},
 {'file': 'careplans.csv',
  'field_count': 9,
  'fields': ['Id',
   'START',
   'STOP',
   'PATIENT',
   'ENCOUNTER',
   'CODE',
   'DESCRIPTION',
   'REASONCODE',
   'REASONDESCRIPTION'],
  'wrapper_rows': 27,
  'source_checkout_rows': 27,
  'same_row_count': True,
  'same_sha256': False},
 {'file': 'claims.csv',
  'field_count': 31,
  'fields': ['Id',
   'PATIENTID',
   'PROVIDERID',
   'PRIMARYPATIENTINSURANCEID',
   'SECONDARYPATIENTINSURANCEID',
   'DEPARTMENTID',
   'PATIENTDEPARTMENTID',
   'DIAGNOSIS1',
   'DIAGNOSIS2',
   'DIAGNOSIS3',
   'DIAGNOSIS4',
   'DIAGNOSIS5',
   'DIAGNOSIS6',
   'DIAGNOSIS

## What the equivalence result proves

For release `v4.0.0` and the tested inputs, PyHealth's pinned release JAR preserves the CSV schema contract produced by independently cloning the official source tag and running its `./run_synthea` CLI:

1. the same CSV tables are emitted;
2. every table has the same ordered output fields.

Row counts, patient identifiers, and SHA-256 hashes are diagnostics rather than schema assertions. The same seeds reproduce a run within a fixed executable, but they do not by themselves guarantee identical clinical histories between the published release JAR and a fresh Gradle build of the source tag. This is a schema integration equivalence test, not a byte-for-byte reproducibility claim.